In [3]:
from Math500_bench import MATH500, MATH500_Bench
try:
    import sympy as sp
    HAVE_SYMPY = True
except Exception:
    HAVE_SYMPY = False
import re

c:\Users\kachr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
x = MATH500()


In [5]:
for i,e in enumerate(x):
    print(i)
    print(e['question'])
    print(e['answer'])
    break

0
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
\left( 3, \frac{\pi}{2} \right)


In [6]:
def _extract_boxed(text: str):
        """
        Prefer the last occurrence of \boxed{...}.
        Accept optional surrounding $ ... $ or \( ... \).
        """
        if not text:
            return None
        # 1) match optional $ or \( around \boxed{...}
        boxed_matches = re.findall(r'\\boxed\{((?:[^{}]|{[^{}]*})*)\}', text, flags=re.DOTALL)
        if boxed_matches:
            return boxed_matches
    
        return None
def _latex_normalize(s: str):
        """
        Lightweight normalization of LaTeX-ish final answers to a canonical string.
        - strips wrapping $...$ or \( ... \)
        - removes \left/\right and thin spaces
        - converts simple \frac{a}{b} -> (a)/(b)
        - collapses whitespace
        Returns normalized string (or None).
        """
        if s is None:
            return None
        s = s.strip()
        # strip surrounding $ or \( \) or \( ... \) occurrences
        s = re.sub(r"^\$+|\\begin\{.*?\}|\\end\{.*?\}|\$+$", "", s)
        s = s.strip()
        # remove surrounding \( \) or \[ \]
        s = re.sub(r"^\\\(|\\\)$", "", s)
        s = s.strip()
        # remove \left and \right
        s = re.sub(r"\\left|\\right", "", s)
        # convert \pi to "pi" (so both sides are consistent)
        s = s.replace(r"\pi", "pi")
        # convert \frac{a}{b} to (a)/(b)
        s = re.sub(r"\\frac\{\s*([^\{\}]+?)\s*\}\{\s*([^\{\}]+?)\s*\}", r"(\1)/(\2)", s)
        # remove redundant whitespace
        s = re.sub(r"\s+", "", s)
        return s
def _canonicalize(s: str):
        """
        Try to create a canonical representation. If sympy is available, attempt symbolic simplification.
        Otherwise return the normalized string.
        """
        if s is None:
            return None
        norm = _latex_normalize(s)
        if norm is None:
            return None
    
        if HAVE_SYMPY:
            try:
                # Replace caret with ** for pow if present
                norm_for_sympy = norm.replace("^", "**")
                expr = sp.sympify(norm_for_sympy)
                simplified = sp.simplify(expr)
                return simplified  # sympy object
            except Exception:
                # fallback to normalized string
                return norm
        else:
            return norm


In [7]:
def sympy_checker(expr1, expr2):
    is_correct = False
    if HAVE_SYMPY and isinstance(expr1, sp.Expr) and isinstance(expr2, sp.Expr):
        try:
            is_correct = (sp.simplify(expr1 - expr2) == 0) 
        except Exception:
            is_correct = (str(expr1) == str(expr2))
    else:
        is_correct = (str(expr1) == str(expr2))
    return is_correct

In [8]:
cols = ["question", "gold", "pred_text", "pred_raw", "extract_method", "pred_norm", "gold_norm", "correct"]

In [9]:
results = []
with open('test_sample_output.jsonl') as f:
    for i,line in enumerate(f.readlines()):
        line = line.strip()
        removed_brackets = line[1:-1]
        cleaned_backslashes = removed_brackets.replace('\\\\', '\\')
        cleaned_brackets = cleaned_backslashes.replace('\\(', '(').replace('\\)', ')')
        y = []
        for x in cols:
            y.append(cleaned_brackets.find(f"'{x}':"))
        y.append(len(cleaned_brackets))
        output = {}
        for e in range(len(y)-1):
            key_val = cleaned_brackets[y[e]:y[e+1]]
            key,val = key_val.split(':', 1)
            output[key.strip().strip("'")] = val.strip().strip("',").strip('"')
        results.append(output)

In [12]:
correct, partial = 0,0
for r in results:
    print("pred text: ",r['pred_text'])
    print("gold:", r['gold'], "-->", r['gold_norm'])
    normalized = _latex_normalize(r['pred_text'])
    print("pred text normalized: ", normalized)
    extracted = _extract_boxed(normalized)
    print("pred text extracted boxed instances:", extracted)
    if extracted:
        canonical = list(map(_canonicalize,extracted))
        print(*map(lambda x:isinstance(x,sp.Expr),canonical))
        print("pred text extracted boxed instances in canonical form:",canonical)
    is_correct = sympy_checker(canonical[-1], r['gold_norm']) if canonical else False
    is_partially_correct = any(sympy_checker(pred, r['gold_norm']) for pred in canonical)
    print("is correct:", is_correct, "is partially correct:", is_partially_correct)

    correct += int(is_correct)
    partial += int(is_partially_correct)
    print()
print("running correct/partial:", correct, partial, "out of ", len(results))


pred text:  $\boxed{\left(\sqrt{10},\frac{\pi}{4}\right)}$ To convert from rectangular coordinates $(x,y)$ to polar coordinates $(r,\theta)$ we use the formulas:\n$$ r = \sqrt{x^2 + y^2} $$\n$$ \theta = \arctan\left(\frac{y}{x}\right) $$\n\nFor the given point $(0,3)$:\n\n- The value of $x$ is $0$, so $r = \sqrt{0^2 + 3^2} = \sqrt{9} = 3$.\n- The value of $y$ is $3$, so $\theta = \arctan\left(\frac{3}{0}\right)$. Since the tangent function approaches infinity when divided by zero (from the right), this means that $\theta$ must be $\frac{\pi}{2}$ radians or $90^\circ$.\n\nTherefore, the polar coordinates are $\boxed{(3,\frac{\pi}{2})}$. However, since the question asks for a solution with $r > 0$ and $0 \leq \theta < 2\pi$, we can express it as $\boxed{\left(3,\frac{\pi}{2}\right)}$. But if you want to keep it consistent with the original format provided, we should note that the angle $\theta$ cannot be exactly $\frac{\pi}{2}$ because division by zero is undefined. So, the correct repre